# Comparação do pipeline: aorta e óstios

Avaliação operacional do **novo padrão: filtro + envelope + lower100/pad2**,
comparado ao P99.9 puro e à correção da aorta sem padding. Mantém -300 HU,
P99.9, RG e o pós-processamento arterial. O filtro usa geometria 4.8/8 mm,
cinco círculos sintéticos, envelope 2.25r/margem 10 e level set b0.6/r0.10/i26.

A tabela principal usa treino (30), validação (270) e teste (700), sem juntar
subconjuntos. A revisão visual histórica (30/60) aparece separadamente ao final:
sucesso automático dos óstios não equivale a qualidade visual da aorta.
Nenhum pipeline pesado ou HTML é gerado aqui; os dados vêm dos runs salvos.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# Localiza a raiz antes de importar os módulos do projeto.
current = Path.cwd().resolve()
REPO_ROOT = next(
    path
    for path in [current, *current.parents]
    if (path / "src").exists() and (path / "output").exists()
)
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from utils.experiments import (  # noqa: E402
    get_aorta_visual_review,
    load_aorta_visual_reviews,
    resolve_aorta_review_summary_path,
)
from utils.project.notebook_env import configure_notebook_environment  # noqa: E402

REPO_ROOT = configure_notebook_environment(chdir_to_src=False)
pd.set_option("display.max_columns", 40)
from utils.comparison_utils.paired_statistics import (  # noqa: E402
    adjust_holm,
    compare_paired_dice,
)

## 1. Runs e coortes

Caminhos explícitos: não seleciona silenciosamente o run mais recente.


In [ ]:
RUNS = {
    "train": {
        "puro": "output/segmentation/runs/mid_res/current_baseline_p99_9/train/2026-08-06_18-43-37",
        "filtro_envelope": "output/segmentation/runs/mid_res/aorta_segmentation_experiments/train/trajectory_geometry_r4_8_c8_0_p99_9_m300/2026-09-05_19-24-41",
        "filtro_envelope_pad2": "output/segmentation/runs/mid_res/aorta_segmentation_experiments/train/ostia_localization/lower100_pad2/2026-09-06_08-08-25",
    },
    "val": {
        "puro": "output/segmentation/runs/mid_res/current_baseline_p99_9/val/2026-08-06_22-43-14",
        "filtro_envelope": "output/segmentation/runs/mid_res/aorta_segmentation_experiments/val/ostia_validation270/baseline_pad0/2026-09-07_07-15-02",
        "filtro_envelope_pad2": "output/segmentation/runs/mid_res/aorta_segmentation_experiments/val/ostia_validation270/lower100_pad2/2026-09-07_07-15-15",
    },
    "test": {
        "puro": "output/segmentation/runs/mid_res/current_baseline_p99_9/test/2026-08-06_10-04-22",
        "filtro_envelope": "output/segmentation/runs/mid_res/aorta_segmentation_experiments/test/ostia_comparison/baseline_pad0/2026-09-07_10-37-23",
        "filtro_envelope_pad2": "output/segmentation/runs/mid_res/aorta_segmentation_experiments/test/ostia_comparison/lower100_pad2/2026-09-07_10-37-38",
    },
}
EXPECTED_IMAGES = {"train": 30, "val": 270, "test": 700}
COHORT_NAMES = {"train": "Treino", "val": "Validação", "test": "Teste"}
METHOD_NAMES = {
    "puro": "P99.9 puro (histórico)",
    "filtro_envelope": "Filtro + envelope / pad0",
    "filtro_envelope_pad2": "Filtro + envelope / pad2 (padrão)",
}
SUCCESS = {
    "both correct",
    "both tolerable",
    "both ostia correct",
    "both ostia tolerable",
}
run_frames = {}
for split, variants in RUNS.items():
    for variant, directory in variants.items():
        path = REPO_ROOT / directory / "numeric" / f"ostios_{split}_summary.csv"
        frame = pd.read_csv(path)
        frame["IMG_ID"] = pd.to_numeric(frame["IMG_ID"], errors="raise").astype(int)
        if frame["IMG_ID"].duplicated().any() or len(frame) != EXPECTED_IMAGES[split]:
            raise ValueError(f"Coorte incompleta/duplicada: {split}/{variant}")
        status = (
            frame["ostia_detection_status"]
            .astype(str)
            .str.lower()
            .str.replace("_", " ")
            .str.strip()
        )
        frame["ostia_success"] = status.isin(SUCCESS)
        frame["artery_dice"] = pd.to_numeric(frame["artery_dice"], errors="raise")
        if not frame["artery_dice"].between(0, 1).all():
            raise ValueError(f"Dice inválido/ausente: {split}/{variant}")
        run_frames[split, variant] = frame
    ids = set(run_frames[split, "puro"]["IMG_ID"])
    if any(set(run_frames[split, v]["IMG_ID"]) != ids for v in variants):
        raise ValueError(f"IDs diferentes em {split}")

## 2. Desempenho completo

Both correct e both tolerable contam igualmente como sucesso. A média condicionada usa subconjuntos diferentes e não substitui a comparação pareada geral.


In [ ]:
# Preserva Dice zero e falhas na média geral.
rows = []
for (split, variant), frame in run_frames.items():
    valid = frame["ostia_success"]
    rows.append(
        {
            "subconjunto": COHORT_NAMES[split],
            "variante": METHOD_NAMES[variant],
            "exames": len(frame),
            "óstios_sucesso": int(valid.sum()),
            "óstios_sucesso_%": 100 * valid.mean(),
            "dice_médio_todos": frame["artery_dice"].mean(),
            "dice_mediano": frame["artery_dice"].median(),
            "dice_std": frame["artery_dice"].std(),
            "dice_médio_óstios_válidos": frame.loc[valid, "artery_dice"].mean(),
            "dice_zero": int(frame["artery_dice"].eq(0).sum()),
        }
    )
pipeline_overview = pd.DataFrame(rows)
display(pipeline_overview.round(4))

## 3. Comparações pareadas e Wilcoxon

Delta = candidato menos referência. Wilcoxon bilateral (zero_method=wilcox,
method=auto), com deltas arredondados a 12 casas. Exames com Dice zero permanecem;
apenas deltas exatamente zero não entram nos ranks. P ajustado por Holm nas nove
comparações. Efeito rank-biserial positivo favorece o candidato, mas não mede
o ganho médio de Dice. Consulte também mediana, perdas e ganhos por exame.

Wilcoxon testa a distribuição das diferenças sob hipótese de simetria, não a
média de Dice diretamente. Resultados exploratórios de treino/validação e
confirmação no teste ficam separados; não demonstram ausência de regressões
nem validam a qualidade visual das 700 aortas.


In [ ]:
# A família inclui três comparações por split; Holm é aplicado às nove.
pair_rows = []
for split in RUNS:
    for reference, candidate in (
        ("puro", "filtro_envelope"),
        ("puro", "filtro_envelope_pad2"),
        ("filtro_envelope", "filtro_envelope_pad2"),
    ):
        ref = run_frames[split, reference]
        cand = run_frames[split, candidate]
        stats = compare_paired_dice(ref, cand)
        ostia = (
            ref.set_index("IMG_ID")["ostia_success"]
            .to_frame("reference")
            .join(
                cand.set_index("IMG_ID")["ostia_success"].rename("candidate"),
                how="inner",
            )
        )
        pair_rows.append(
            {
                "subconjunto": COHORT_NAMES[split],
                "referência": METHOD_NAMES[reference],
                "candidato": METHOD_NAMES[candidate],
                **stats,
                "óstios_recuperados": int(
                    (~ostia["reference"] & ostia["candidate"]).sum()
                ),
                "óstios_perdidos": int(
                    (ostia["reference"] & ~ostia["candidate"]).sum()
                ),
            }
        )
paired_statistics = pd.DataFrame(pair_rows)
paired_statistics["p_holm"] = adjust_holm(paired_statistics["p_value"])
paired_statistics["significativo_005"] = paired_statistics["p_holm"].lt(0.05)
with pd.option_context("display.float_format", "{:.6g}".format):
    display(paired_statistics)

## 4. Métricas por subconjunto


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(13, 11), constrained_layout=True)
for row, split in enumerate(RUNS):
    data = pipeline_overview[pipeline_overview["subconjunto"].eq(COHORT_NAMES[split])]
    for col, (metric, label, limit) in enumerate(
        (
            ("dice_médio_todos", "Dice médio", 1.1),
            ("óstios_sucesso_%", "Sucesso dos óstios (%)", 110),
        )
    ):
        ax = axes[row, col]
        bars = ax.bar(
            ["Puro", "Filtro + envelope", "+ pad2"],
            data[metric],
            color=["#4C78A8", "#54A24B", "#E45756"],
        )
        ax.set_title(COHORT_NAMES[split])
        ax.set_ylabel(label)
        ax.set_ylim(0, limit)
        ax.bar_label(bars, fmt="%.3f" if col == 0 else "%.1f", padding=4)
        ax.grid(axis="y", alpha=0.2)
plt.show()

## 5. Revisão visual histórica (30 treino / 60 validação)

As variantes abaixo são as versões realmente inspecionadas, não as 270/700 imagens. Os runs históricos podem diferir na geometria ou em correções posteriormente removidas. Seus rótulos não são transferidos automaticamente ao novo padrão pad2.


### 5.1. Configuração e carregamento

Os caminhos dos runs e as classificações visuais ficam centralizados em `config/aorta_visual_reviews.json`. A comparação usa apenas variantes disponíveis nas mesmas coortes de 30 imagens de treino e 60 de validação.


In [ ]:
REVIEW_CONFIG_PATH = REPO_ROOT / "config/aorta_visual_reviews.json"
SPLITS = ("train", "val")
VARIANTS = (
    "normal",
    "filter_envelope_current",
    "levelset_b0_6_r0_10_i26",
)

SPLIT_NAMES = {"train": "Treino", "val": "Validação"}
DISPLAY_NAMES = {
    "normal": "Baseline",
    "filter_envelope_current": "Filtro + envelope",
    "levelset_b0_6_r0_10_i26": "Filtro + envelope + level set refinado",
}
SUCCESS_STATUSES = {
    "both correct",
    "both tolerable",
    "both ostia correct",
    "both ostia tolerable",
}
SUMMARY_COLUMNS = [
    "IMG_ID",
    "artery_dice",
    "ostia_detection_status",
    "aorta_mask_voxel_count",
    "aorta_volume_fraction",
]

review_catalog = load_aorta_visual_reviews(REVIEW_CONFIG_PATH)
reviews = {
    (split, variant): get_aorta_visual_review(review_catalog, variant, split)
    for split in SPLITS
    for variant in VARIANTS
}


def load_solution(split, variant):
    """Carrega métricas automáticas e acrescenta o rótulo visual da aorta."""
    review = reviews[(split, variant)]
    summary_path = resolve_aorta_review_summary_path(REPO_ROOT, review, split)
    frame = pd.read_csv(summary_path)

    missing_columns = set(SUMMARY_COLUMNS).difference(frame.columns)
    if missing_columns:
        raise ValueError(
            f"Colunas ausentes em {variant}/{split}: {sorted(missing_columns)}"
        )

    # Mantém somente as métricas compartilhadas pelos três experimentos.
    frame = frame[SUMMARY_COLUMNS].copy()
    frame["IMG_ID"] = pd.to_numeric(frame["IMG_ID"], errors="raise").astype(int)

    expected_ids = review["aorta_good_ids"] | review["aorta_bad_ids"]
    observed_ids = set(frame["IMG_ID"])
    if observed_ids != expected_ids:
        raise ValueError(
            f"IDs incompatíveis em {variant}/{split}: "
            f"ausentes={sorted(expected_ids - observed_ids)}; "
            f"não revisados={sorted(observed_ids - expected_ids)}"
        )

    # Both correct e both tolerable representam sucesso dos dois óstios.
    normalized_status = (
        frame["ostia_detection_status"]
        .astype(str)
        .str.lower()
        .str.replace("_", " ", regex=False)
        .str.strip()
    )
    frame["ostia_success"] = normalized_status.isin(SUCCESS_STATUSES)
    frame["aorta_visual_good"] = frame["IMG_ID"].isin(review["aorta_good_ids"])
    frame["variant"] = variant
    frame["split"] = split
    return frame


solutions = {
    (split, variant): load_solution(split, variant)
    for split in SPLITS
    for variant in VARIANTS
}

for split in SPLITS:
    cohort_ids = [set(solutions[(split, variant)]["IMG_ID"]) for variant in VARIANTS]
    if any(ids != cohort_ids[0] for ids in cohort_ids[1:]):
        raise ValueError(f"As variantes de {split} não usam a mesma coorte.")
    print(f"{SPLIT_NAMES[split]}: {len(cohort_ids[0])} exames em cada variante")

### 5.2. Resultados gerais

A tabela reúne qualidade visual da aorta, sucesso automático dos dois óstios e Dice da segmentação arterial. O volume médio é mantido como apoio para identificar alterações globais na máscara.


In [ ]:
overview_rows = []
for split in SPLITS:
    for variant in VARIANTS:
        frame = solutions[(split, variant)]
        overview_rows.append(
            {
                "subconjunto": SPLIT_NAMES[split],
                "solução": DISPLAY_NAMES[variant],
                "imagens": len(frame),
                "aortas_boas": int(frame["aorta_visual_good"].sum()),
                "aortas_boas_%": 100 * frame["aorta_visual_good"].mean(),
                "sucesso_óstios_csv": int(frame["ostia_success"].sum()),
                "sucesso_óstios_%": 100 * frame["ostia_success"].mean(),
                "dice_médio": frame["artery_dice"].mean(),
                "dice_mediano": frame["artery_dice"].median(),
                "volume_aorta_médio_%": 100 * frame["aorta_volume_fraction"].mean(),
            }
        )

overview_df = pd.DataFrame(overview_rows)
display(overview_df.round(4))

### 5.3. Diferença em relação ao baseline

Os deltas são calculados de forma pareada pelos mesmos `IMG_IDs`. Valores positivos em Dice e sucesso dos óstios favorecem a variante. As listas de aortas corrigidas e pioradas vêm exclusivamente da inspeção visual.


In [ ]:
comparison_rows = []
for split in SPLITS:
    baseline = solutions[(split, "normal")].set_index("IMG_ID").sort_index()
    baseline_review = reviews[(split, "normal")]

    for variant in VARIANTS[1:]:
        candidate = solutions[(split, variant)].set_index("IMG_ID").sort_index()
        candidate_review = reviews[(split, variant)]
        paired_ids = baseline.index.intersection(candidate.index)

        normal_bad = baseline_review["aorta_bad_ids"]
        candidate_bad = candidate_review["aorta_bad_ids"]
        comparison_rows.append(
            {
                "subconjunto": SPLIT_NAMES[split],
                "solução": DISPLAY_NAMES[variant],
                "delta_dice_médio": (
                    candidate.loc[paired_ids, "artery_dice"]
                    - baseline.loc[paired_ids, "artery_dice"]
                ).mean(),
                "delta_sucesso_óstios_pp": 100
                * (
                    candidate.loc[paired_ids, "ostia_success"].mean()
                    - baseline.loc[paired_ids, "ostia_success"].mean()
                ),
                "delta_aortas_boas": (
                    int(candidate.loc[paired_ids, "aorta_visual_good"].sum())
                    - int(baseline.loc[paired_ids, "aorta_visual_good"].sum())
                ),
                "máscaras_com_voxels_diferentes": int(
                    (
                        candidate.loc[paired_ids, "aorta_mask_voxel_count"]
                        != baseline.loc[paired_ids, "aorta_mask_voxel_count"]
                    ).sum()
                ),
                "aortas_corrigidas": sorted(normal_bad - candidate_bad),
                "aortas_pioradas": sorted(candidate_bad - normal_bad),
            }
        )

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df.round(4))

### 5.4. Comparação visual das métricas

Cada linha representa um subconjunto. As escalas percentuais da qualidade da aorta e dos óstios são separadas do Dice para evitar interpretações equivocadas.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8), constrained_layout=True)
colors = ["#4C78A8", "#54A24B", "#E45756"]
metrics = (
    ("aortas_boas_%", "Aortas visualmente boas (%)", (0, 110)),
    ("sucesso_óstios_%", "Sucesso dos óstios pelo CSV (%)", (0, 110)),
    ("dice_médio", "Dice médio", (0, 1.08)),
)

for row, split in enumerate(SPLITS):
    split_df = overview_df[overview_df["subconjunto"].eq(SPLIT_NAMES[split])]
    for column, (metric, title, limits) in enumerate(metrics):
        axis = axes[row, column]
        bars = axis.bar(split_df["solução"], split_df[metric], color=colors)
        axis.set_title(f"{SPLIT_NAMES[split]}: {title}", fontsize=11)
        axis.set_ylim(*limits)
        axis.tick_params(axis="x", rotation=18, labelsize=9)
        axis.grid(axis="y", alpha=0.25)

        decimals = 3 if metric == "dice_médio" else 1
        labels = [f"{value:.{decimals}f}" for value in split_df[metric]]
        axis.bar_label(bars, labels=labels, padding=3, fontsize=9)

plt.show()

### 5.5. Síntese

A melhor configuração pode variar conforme o desfecho. Esta síntese informa separadamente o maior Dice, a melhor qualidade visual da aorta e a maior taxa de sucesso dos óstios em cada subconjunto.


In [ ]:
for split in SPLITS:
    split_df = overview_df[overview_df["subconjunto"].eq(SPLIT_NAMES[split])]
    best_dice = split_df.loc[split_df["dice_médio"].idxmax()]
    best_aorta = split_df.loc[split_df["aortas_boas_%"].idxmax()]
    best_ostia = split_df.loc[split_df["sucesso_óstios_%"].idxmax()]

    print(SPLIT_NAMES[split])
    print(f"- Maior Dice: {best_dice['solução']} ({best_dice['dice_médio']:.4f})")
    print(
        f"- Mais aortas visualmente boas: {best_aorta['solução']} "
        f"({best_aorta['aortas_boas_%']:.1f}%)"
    )
    print(
        f"- Maior sucesso dos óstios: {best_ostia['solução']} "
        f"({best_ostia['sucesso_óstios_%']:.1f}%)"
    )

## 6. Decisão operacional

O padrão passa a ser filtro + envelope + lower100/pad2: o Dice e o sucesso dos óstios
aumentaram em treino, validação e teste frente ao P99.9 puro. No teste, o Dice
passou de 0,5930 para 0,6018 e o sucesso de 578/700 para 599/700.
Isso não representa melhora uniforme: 57 exames recuperaram sucesso e 36 perderam.
O Wilcoxon contra o puro não foi significativo (p bruto 0,172 no teste e 0,353
na validação; também não após Holm). A promoção é operacional, não uma afirmação
de superioridade estatística de Dice. A revisão visual dos novos runs completos
continua pendente; as anotações históricas 30/60 não são extrapoladas.
